# FLEURS Welsh — data explorationDataset stats and the zero-shot Whisper-small baseline that the fine-tunedmodel has to beat. Heavy work lives in `src/`; this notebook just looks at things.

In [ ]:
import sys, collections, jsonsys.path.insert(0, "../src")import numpy as npimport matplotlib.pyplot as pltfrom data import load_split, normalize_text, has_digit, SAMPLE_RATE

## Split sizes and duration

In [ ]:
splits = {s: load_split(s) for s in ["train", "validation", "test"]}dur = {s: np.array(d["num_samples"]) / SAMPLE_RATE for s, d in splits.items()}for s, d in splits.items():    x = dur[s]    print(f"{s:11s} n={len(d):5d}  hours={x.sum()/3600:6.2f}  "          f"mean={x.mean():5.2f}s  median={np.median(x):5.2f}s  max={x.max():6.2f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))ax.hist(dur["train"], bins=50, color="#4C72B0", edgecolor="white")ax.axvline(30, color="crimson", ls="--", label="30s training filter")ax.set_xlabel("utterance duration (s)"); ax.set_ylabel("count")ax.set_title("FLEURS Welsh train — utterance duration"); ax.legend()plt.tight_layout(); plt.show()print("share of train dropped by a 30s filter:", f"{(dur['train'] > 30).mean():.3%}")print("share of train dropped by a 20s filter:", f"{(dur['train'] > 20).mean():.3%}",      f"({dur['train'][dur['train'] > 20].sum()/3600:.2f}h)")

The 20s cutoff suggested by most XLS-R tutorials would discard ~3.7 of 12.2training hours here. FLEURS utterances are long (median ~12.7s), so we filterat 30s instead and control memory with `group_by_length` plus gradient accumulation.

## Character inventory

In [ ]:
raw_chars = collections.Counter("".join(splits["train"]["transcription"]))norm_chars = collections.Counter("".join(normalize_text(t) for t in splits["train"]["transcription"]))print(f"raw:        {len(raw_chars):3d} distinct -> {''.join(sorted(raw_chars))}")print()print(f"normalized: {len(norm_chars):3d} distinct -> {''.join(sorted(norm_chars))}")

Normalization lowercases, deletes apostrophes (`mae'r` → `maer`), stripspunctuation, keeps the Welsh circumflex vowels `âêîôûŵŷ`, and folds foreigndiacritics (`é`, `ç`, `ü`) onto their base letters — those appear only a handfulof times in names and would otherwise add tokens with almost no training signal.

## Numerals

In [ ]:
for s in ["train", "test"]:    t = splits[s]["transcription"]    n = sum(has_digit(x) for x in t)    print(f"{s:5s}: {n:4d}/{len(t)} utterances contain digits ({n/len(t):.1%})")for t in splits["train"]["transcription"]:    if has_digit(t):        print("\nexample:", normalize_text(t)[:110]); break

Roughly a fifth of utterances contain numerals. A character-level CTC model hasno way to map the audio of *dwy fil ar bymtheg* onto the characters `2019`, sothese are close to unlearnable. We keep them (dropping them would cost ~22% of a12-hour corpus) and report WER both overall and on the digit-free subset — thesame two subsets for both models, so the comparison stays internally valid.

## Zero-shot Whisper-small baseline

In [ ]:
# Produced by: python src/evaluate.py --out results/whisper_baseline.jsonbase = json.load(open("../results/whisper_baseline.json"))print(f"overall    WER {base['overall']['wer']:.4f}   CER {base['overall']['cer']:.4f}   n={base['overall']['n']}")print(f"no digits  WER {base['no_digits']['wer']:.4f}   CER {base['no_digits']['cer']:.4f}   n={base['no_digits']['n']}")

In [ ]:
for r in base["predictions"][:5]:    print("REF :", r["ref"][:100])    print("PRED:", r["pred"][:100])    print()